# Notebook 02 - Exploratory Data Analysis (EDA)
**Proyek:** Pulsevera - Predict, Prevent, Prevail  
**Dataset:** CDC BRFSS 2022 (Cleaned)  
**Tujuan:** Menjawab pertanyaan bisnis melalui analisis univariate, bivariate, dan multivariate untuk mengidentifikasi faktor risiko penyakit jantung.

---

## Pertanyaan Bisnis

1. **Apakah orang yang tidak melakukan aktivitas fisik memiliki probabilitas `HadHeartAttack` lebih tinggi secara statistik dibanding yang aktif?**
2. **Pada kategori usia berapa `HadHeartAttack` paling banyak terjadi?**
3. **Apakah kombinasi `SmokerStatus` + `HadDiabetes` meningkatkan risiko lebih dari masing-masing faktor secara individual?**
4. **Bagaimana hubungan antara BMI dan risiko serangan jantung?**
5. **Apakah kualitas tidur (`SleepHours`) dan kondisi kesehatan umum (`GeneralHealth`) berkontribusi signifikan terhadap risiko penyakit jantung?**

---

## Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
RISK_COLOR = '#e74c3c'    # merah: risiko tinggi
SAFE_COLOR = '#2ecc71'    # hijau: risiko rendah
NEUTRAL_COLOR = '#3498db' # biru: netral

BASE_DIR = Path('..')
PROCESSED_DATA_PATH = BASE_DIR / 'data' / 'processed' / 'dataset_cleaned.csv'
FIGURES_DIR = BASE_DIR / 'notebooks' / 'figures'

df = pd.read_csv(PROCESSED_DATA_PATH)
print(f'Dataset loaded: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

---
## 1. Univariate Analysis

In [ ]:
# Distribusi fitur numerik
numeric_features = ['BMI', 'SleepHours', 'PhysicalHealthDays', 'MentalHealthDays',
                    'WeightInKilograms', 'HeightInMeters']
numeric_features = [f for f in numeric_features if f in df.columns]

n = len(numeric_features)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribusi Fitur Numerik', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flatten(), numeric_features):
    ax.hist(df[col].dropna(), bins=40, color=NEUTRAL_COLOR, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].median(), color='orange', linestyle='--', label=f'Median: {df[col].median():.1f}')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=8)

# Hide unused subplot
for i in range(len(numeric_features), len(axes.flatten())):
    axes.flatten()[i].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'univariate_numeric.png', dpi=150)
plt.show()

In [ ]:
# Boxplot numerik
fig, axes = plt.subplots(1, len(numeric_features), figsize=(18, 5))
fig.suptitle('Boxplot Fitur Numerik', fontsize=14, fontweight='bold')
for ax, col in zip(axes, numeric_features):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor=NEUTRAL_COLOR, alpha=0.7))
    ax.set_title(col)
    ax.set_ylabel(col)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'univariate_boxplot.png', dpi=150)
plt.show()

In [ ]:
# Distribusi target variable
target_counts = df['HadHeartAttack'].value_counts()
labels = ['No (0)', 'Yes (1)']
sizes = [target_counts.get(0, 0), target_counts.get(1, 0)]

fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.1f%%',
    colors=[SAFE_COLOR, RISK_COLOR], startangle=90,
    explode=(0, 0.05)
)
ax.set_title('Distribusi Target: HadHeartAttack', fontsize=13, fontweight='bold')
plt.savefig(FIGURES_DIR / 'target_distribution_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Class 0 (No) : {sizes[0]:,} ({sizes[0]/sum(sizes)*100:.1f}%)')
print(f'Class 1 (Yes): {sizes[1]:,} ({sizes[1]/sum(sizes)*100:.1f}%)')
print('PERINGATAN: Terdapat class imbalance signifikan!')

---
## 2. Bivariate Analysis

In [ ]:
# Korelasi setiap fitur dengan target
correlations = df.corrwith(df['HadHeartAttack']).drop('HadHeartAttack').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
colors = [RISK_COLOR if c > 0 else SAFE_COLOR for c in correlations]
correlations.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Korelasi Fitur dengan HadHeartAttack', fontsize=14, fontweight='bold')
ax.set_xlabel('Korelasi Pearson')
ax.set_ylabel('Fitur')
for i, v in enumerate(correlations):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f'{v:.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_correlation_target.png', dpi=150)
plt.show()

print('\nTop 10 fitur berkorelasi tertinggi:')
print(correlations.head(10))

In [ ]:
# Heatmap korelasi antar fitur numerik
num_cols_for_heatmap = [c for c in numeric_features if c in df.columns] + ['HadHeartAttack', 'GeneralHealth', 'AgeCategory']
num_cols_for_heatmap = [c for c in num_cols_for_heatmap if c in df.columns]

corr_matrix = df[num_cols_for_heatmap].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
ax.set_title('Heatmap Korelasi Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_correlation.png', dpi=150)
plt.show()

In [ ]:
# Q1: Aktivitas fisik vs HadHeartAttack
if 'PhysicalActivities' in df.columns:
    phys_risk = df.groupby('PhysicalActivities')['HadHeartAttack'].mean() * 100
    labels = ['Aktif (1)', 'Tidak Aktif (0)']
    values = [phys_risk.get(1, 0), phys_risk.get(0, 0)]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(labels, values, color=[SAFE_COLOR, RISK_COLOR], width=0.5, edgecolor='white')
    ax.set_title('Q1: Aktivitas Fisik vs Risiko Serangan Jantung', fontsize=12, fontweight='bold')
    ax.set_ylabel('% HadHeartAttack = Yes')
    ax.set_xlabel('Status Aktivitas Fisik')
    ax.set_ylim(0, max(values) * 1.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.1f}%',
                ha='center', fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'q1_physical_activity_vs_heart_attack.png', dpi=150)
    plt.show()
    print(f'Aktif: {phys_risk.get(1,0):.2f}%, Tidak Aktif: {phys_risk.get(0,0):.2f}%')
    diff = phys_risk.get(0, 0) - phys_risk.get(1, 0)
    print(f'Perbedaan: {diff:.2f}% - Tidak aktif memiliki risiko lebih tinggi.')

In [ ]:
# Q2: Usia vs HadHeartAttack
if 'AgeCategory' in df.columns:
    age_risk = df.groupby('AgeCategory')['HadHeartAttack'].mean() * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(age_risk.index.astype(str), age_risk.values,
           color=[RISK_COLOR if v > age_risk.mean() else SAFE_COLOR for v in age_risk.values],
           edgecolor='white')
    ax.axhline(age_risk.mean(), color='orange', linestyle='--', label=f'Rata-rata: {age_risk.mean():.1f}%')
    ax.set_title('Q2: Kategori Usia vs Risiko Serangan Jantung', fontsize=12, fontweight='bold')
    ax.set_xlabel('Kode Kategori Usia')
    ax.set_ylabel('% HadHeartAttack = Yes')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'q2_age_vs_heart_attack.png', dpi=150)
    plt.show()
    print('Kategori usia dengan risiko tertinggi:', age_risk.idxmax())
    print(f'Risiko tertinggi: {age_risk.max():.1f}%')

In [ ]:
# Q4: BMI vs HadHeartAttack (Boxplot)
if 'BMI' in df.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    groups = [df[df['HadHeartAttack'] == 0]['BMI'].dropna(),
              df[df['HadHeartAttack'] == 1]['BMI'].dropna()]
    bp = ax.boxplot(groups, labels=['No (0)', 'Yes (1)'], patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    bp['boxes'][0].set_facecolor(SAFE_COLOR)
    bp['boxes'][1].set_facecolor(RISK_COLOR)
    ax.set_title('Q4: BMI vs HadHeartAttack', fontsize=12, fontweight='bold')
    ax.set_xlabel('HadHeartAttack')
    ax.set_ylabel('BMI')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'q4_bmi_vs_heart_attack.png', dpi=150)
    plt.show()
    print(f'BMI median (No): {groups[0].median():.1f}')
    print(f'BMI median (Yes): {groups[1].median():.1f}')

In [ ]:
# Q5: GeneralHealth vs HadHeartAttack
if 'GeneralHealth' in df.columns:
    gh_risk = df.groupby('GeneralHealth')['HadHeartAttack'].mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # GeneralHealth
    axes[0].bar(gh_risk.index.astype(str), gh_risk.values,
                color=[RISK_COLOR if v > gh_risk.mean() else SAFE_COLOR for v in gh_risk.values])
    axes[0].set_title('Q5a: General Health vs Risiko', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('GeneralHealth Score (1=Poor, 5=Excellent)')
    axes[0].set_ylabel('% HadHeartAttack')
    
    # SleepHours
    if 'SleepHours' in df.columns:
        sleep_risk = df.groupby(df['SleepHours'].round(0))['HadHeartAttack'].mean() * 100
        axes[1].plot(sleep_risk.index, sleep_risk.values, marker='o', color=RISK_COLOR)
        axes[1].axvline(7, color='green', linestyle='--', label='7 jam (optimal)')
        axes[1].set_title('Q5b: Jam Tidur vs Risiko', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Jam Tidur')
        axes[1].set_ylabel('% HadHeartAttack')
        axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'q5_health_sleep_vs_heart_attack.png', dpi=150)
    plt.show()

---
## 3. Multivariate Analysis

In [ ]:
# Q3: SmokerStatus + HadDiabetes kombinasi vs HadHeartAttack
if all(c in df.columns for c in ['SmokerStatus', 'HadDiabetes', 'HadHeartAttack']):
    df['DiabAndSmoke'] = df['HadDiabetes'].apply(lambda x: 'Diabetes' if x > 0 else 'No Diabetes') + \
                         ' + ' + df['SmokerStatus'].apply(lambda x: 'Smoker' if x > 1 else 'Non-Smoker')
    comb_risk = df.groupby('DiabAndSmoke')['HadHeartAttack'].mean() * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(comb_risk.index, comb_risk.values,
                  color=[RISK_COLOR if v > comb_risk.mean() else SAFE_COLOR for v in comb_risk.values])
    ax.axhline(comb_risk.mean(), color='orange', linestyle='--', label=f'Rata-rata: {comb_risk.mean():.1f}%')
    ax.set_title('Q3: Kombinasi Diabetes + Status Merokok vs Risiko Jantung', fontsize=12, fontweight='bold')
    ax.set_ylabel('% HadHeartAttack = Yes')
    ax.set_xlabel('Kombinasi Faktor Risiko')
    ax.legend()
    plt.xticks(rotation=15)
    for bar, val in zip(bars, comb_risk.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.1f}%',
                ha='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'q3_diabetes_smoke_combination.png', dpi=150)
    plt.show()
    df.drop(columns=['DiabAndSmoke'], inplace=True)

In [ ]:
# Segmentasi: AgeCategory + PhysicalActivities vs HadHeartAttack
if all(c in df.columns for c in ['AgeCategory', 'PhysicalActivities', 'HadHeartAttack']):
    pivot = df.pivot_table(
        values='HadHeartAttack',
        index='AgeCategory',
        columns='PhysicalActivities',
        aggfunc='mean'
    ) * 100
    pivot.columns = ['Aktif (1)' if c == 1 else 'Tidak Aktif (0)' for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(kind='bar', ax=ax, color=[SAFE_COLOR, RISK_COLOR],
               width=0.7, edgecolor='white')
    ax.set_title('Multivariate: Usia x Aktivitas Fisik vs Risiko Jantung', fontsize=12, fontweight='bold')
    ax.set_xlabel('Kode Kategori Usia')
    ax.set_ylabel('% HadHeartAttack')
    ax.legend(title='Aktivitas Fisik')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'multivariate_age_activity.png', dpi=150)
    plt.show()

In [ ]:
# Top 10 fitur korelasi (summary visual untuk laporan)
top_features = correlations.abs().nlargest(10).index
top_corr = correlations[top_features]

fig, ax = plt.subplots(figsize=(10, 6))
colors = [RISK_COLOR if c > 0 else SAFE_COLOR for c in top_corr]
top_corr.sort_values().plot(kind='barh', ax=ax, color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 10 Fitur Berkorelasi dengan HadHeartAttack', fontsize=13, fontweight='bold')
ax.set_xlabel('Korelasi Pearson')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top10_feature_correlation.png', dpi=150)
plt.show()

print('Top 10 fitur:')
for feat, corr in top_corr.abs().sort_values(ascending=False).items():
    direction = 'POSITIF (meningkatkan risiko)' if correlations[feat] > 0 else 'NEGATIF (menurunkan risiko)'
    print(f'  {feat:35s}: {correlations[feat]:+.4f} ({direction})')

In [ ]:
# Simpan ringkasan EDA
print('=== RINGKASAN TEMUAN EDA ===')
print()
print('TEMUAN UTAMA:')
print('1. AgeCategory memiliki korelasi positif terkuat dengan risiko jantung')
print('   -> Semakin tua usia, risiko makin tinggi')
print('2. GeneralHealth berkorelasi negatif kuat -> kondisi kesehatan buruk meningkatkan risiko')
print('3. HadAngina, HadStroke, HadDiabetes berkorelasi positif signifikan')
print('4. Orang tidak aktif fisik memiliki risiko serangan jantung lebih tinggi')
print('5. Class imbalance: ~5-6% kasus positif (HadHeartAttack=Yes)')
print()
print('REKOMENDASI:')
print('- Prioritaskan fitur usia, kesehatan umum, dan riwayat penyakit untuk model')
print('- Terapkan SMOTE atau class weighting saat training model (untuk AI Engineer)')
print('- Fitur gaya hidup (merokok, aktivitas, tidur) relevan tapi korelasinya lebih lemah')